In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.reingresos_30_dias AS

SELECT
    h1.id_paciente,
    h1.id_hospitalizacion AS hospitalizacion_anterior,
    h2.id_hospitalizacion AS hospitalizacion_reingreso,
    h1.fecha_alta AS fecha_alta_anterior,
    h2.fecha_ingreso AS fecha_reingreso,
    DATEDIFF(h2.fecha_ingreso, h1.fecha_alta) AS dias_entre_ingresos,
    h1.motivo AS motivo_anterior,
    h2.motivo AS motivo_reingreso
FROM silver.hospitalizaciones h1
JOIN silver.hospitalizaciones h2
    ON h1.id_paciente = h2.id_paciente
    AND h1.id_hospitalizacion <> h2.id_hospitalizacion
    AND h2.fecha_ingreso > h1.fecha_alta
    AND DATEDIFF(h2.fecha_ingreso, h1.fecha_alta) <= 30;

In [0]:
%sql

SELECT *
FROM gold.reingresos_30_dias
ORDER BY dias_entre_ingresos
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS total_reingresos
FROM gold.reingresos_30_dias;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.medicamentos_recetados AS

SELECT
    r.id_receta,
    r.id_paciente,
    r.id_medico,
    r.fecha_receta,
    rd.id_medicamento,
    m.nombre AS medicamento,
    rd.cantidad
FROM silver.recetas r
JOIN silver.detalle_receta rd
    ON r.id_receta = rd.id_receta
JOIN silver.medicamentos m
    ON rd.id_medicamento = m.id_medicamento;

In [0]:
%sql

SELECT *
FROM gold.medicamentos_recetados
LIMIT 20;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.calidad_aire AS

SELECT
    fecha_hora,
    municipio,
    latitud,
    longitud,
    pm10,
    pm2_5,
    us_aqi,
    CASE
        WHEN us_aqi <= 50 THEN 'Buena'
        WHEN us_aqi <= 100 THEN 'Moderada'
        WHEN us_aqi <= 150 THEN 'No saludable para grupos sensibles'
        WHEN us_aqi <= 200 THEN 'No saludable'
        ELSE 'Muy no saludable'
    END AS categoria_aqi
FROM silver.calidad_aire;

In [0]:
%sql

SELECT *
FROM gold.calidad_aire
LIMIT 20;

In [0]:
%sql

SELECT
    municipio,
    categoria_aqi,
    AVG(pm2_5) AS promedio_pm2_5,
    AVG(us_aqi) AS promedio_aqi
FROM gold.calidad_aire
GROUP BY
    municipio,
    categoria_aqi
ORDER BY promedio_aqi DESC;

In [0]:
# Prueba de conexión con Gitprint("pipeline gold actualizado")